# NEAT proportions vs status quo: objectives vs total panels

This notebook compares **expected objectives** (carbon offset, energy potential, racial equity, income equity) when a total of **T** panels is distributed:

1. **NEAT (Proportional_NEAT)** — ZIP shares match the trained model’s `panel_placements` (renormalized over ZIPs that appear in the federal payload cache).
2. **Status quo** — ZIP shares match **year 5 adoption at $0 federal incentive** under the same state adoption curves and cutoffs as `greedy_neat_match_grouping.ipynb`.

For each **T** from a sweep up to **2,000,000** panels, placements are `T × (ZIP proportion)`; objectives use `create_paper_objectives()` on the full `zips_df` (same as the greedy notebook).

**Requires:** federal `zip_payloads_*.pkl` cache under `Examples/model_cache/federal/` and `Examples/Proportional_NEAT/PROJ_softmax_b0p15_large.pkl`.

In [ ]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "Models").exists():
    repo_root = repo_root.parent

if not (repo_root / "Models").exists():
    raise RuntimeError("Could not locate repository root containing 'Models' directory.")

import sys

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from Data.data_load_util import make_dataset
from Simulation.projections_util import create_paper_objectives

plt.style.use("seaborn-v0_8")

In [ ]:
# --- Config (aligned with greedy_neat_match_grouping) ---
TARGET_ADOPTION_MODE = "additional_only"
YEARS_TO_SIMULATE = 5
STATUS_QUO_YEAR_INDEX = YEARS_TO_SIMULATE - 1  # year 5 (0-based index 4)
REFERENCE_FEDERAL_INCENTIVE_FOR_CALIBRATION = 0

FEDERAL_CACHE_DIR = repo_root / "Examples" / "model_cache" / "federal"
NEAT_PROJ_PATH = repo_root / "Examples" / "Proportional_NEAT" / "PROJ_softmax_b0p15_large.pkl"

MAX_TOTAL_PANELS = 2_000_000
N_SWEEP_POINTS = 80

if not FEDERAL_CACHE_DIR.exists():
    raise RuntimeError("Federal cache directory missing. Build federal payload cache first.")
if not NEAT_PROJ_PATH.exists():
    raise RuntimeError(f"Missing NEAT projection file: {NEAT_PROJ_PATH}")

In [ ]:
state_behavior_df = pd.read_csv(repo_root / "Models" / "Incentives" / "state_behavior.csv")
zips_df, _, _ = make_dataset(granularity="both", remove_outliers=False, load_dir_prefix=str(repo_root / "Data") + "/")
zip_lookup = zips_df.set_index("region_name")
objectives = create_paper_objectives()

with open(NEAT_PROJ_PATH, "rb") as f:
    neat_proj = pickle.load(f)

if not hasattr(neat_proj, "panel_placements"):
    raise RuntimeError("NEAT artifact missing panel_placements")

target_panels = {}
for k, v in neat_proj.panel_placements.items():
    try:
        z = int(k)
    except Exception:
        continue
    if z in zip_lookup.index:
        target_panels[z] = float(v)

target_total = sum(target_panels.values())
target_prop_global = {z: (p / target_total if target_total > 0 else 0.0) for z, p in target_panels.items()}

print(f"NEAT ZIPs in dataset: {len(target_prop_global):,}")
print(f"NEAT total panels (overlap): {target_total:,.0f}")

In [ ]:
def build_target_adoption_curve(base_adoption, annual_increment, years_total, mode):
    out = []
    for year in range(1, years_total + 1):
        if mode == "absolute_total":
            val = base_adoption + year * annual_increment
        else:
            val = year * annual_increment
        out.append(float(np.clip(val, 0.0, 1.0)))
    return out


def required_incentives(payload, cutoff):
    return payload["installation_cost"] - payload["yearly_savings_coeff"] * cutoff


def eval_cutoff(payloads, cutoff, threshold):
    vals = []
    for p in payloads:
        req = required_incentives(p, cutoff)
        vals.append(float(np.mean(req <= threshold)))
    return float(np.mean(vals)) if vals else 0.0


def calibrate_cutoffs(payloads, targets, threshold, lower=0.25, upper=30.0, steps=28):
    cutoffs = []
    prev = lower
    for t in targets:
        lo, hi = prev, upper
        if eval_cutoff(payloads, hi, threshold) < t:
            cutoffs.append(hi)
            prev = hi
            continue
        if eval_cutoff(payloads, lo, threshold) >= t:
            cutoffs.append(lo)
            prev = lo
            continue
        for _ in range(steps):
            mid = 0.5 * (lo + hi)
            if eval_cutoff(payloads, mid, threshold) < t:
                lo = mid
            else:
                hi = mid
        cutoffs.append(hi)
        prev = hi
    return cutoffs


def load_state_payloads(cache_dir):
    state_payloads = {}
    for p in sorted(cache_dir.glob("zip_payloads_*.pkl")):
        parts = p.stem.split("_")
        if len(parts) < 3:
            continue
        st = parts[2]
        with open(p, "rb") as f:
            cached = pickle.load(f)
        payloads = cached.get("payloads", [])
        if payloads:
            if st not in state_payloads or len(payloads) > len(state_payloads[st]):
                state_payloads[st] = payloads
    return state_payloads


state_payloads = load_state_payloads(FEDERAL_CACHE_DIR)
state_cutoffs = {}

for st in tqdm(sorted(state_payloads.keys()), desc="Calibrate state cutoffs", unit="state"):
    row = state_behavior_df[state_behavior_df["State code"] == st]
    if row.empty:
        continue
    row = row.iloc[0]
    curve = build_target_adoption_curve(
        float(row["prop_adopted_status_quo"][1:-1]),
        float(row["prop_adopted_per_year_average"]),
        YEARS_TO_SIMULATE,
        TARGET_ADOPTION_MODE,
    )
    th = -REFERENCE_FEDERAL_INCENTIVE_FOR_CALIBRATION
    state_cutoffs[st] = calibrate_cutoffs(state_payloads[st], curve, th)

print(f"Calibrated states: {len(state_cutoffs)}")

In [ ]:
# ZIP universe: same overlap as greedy notebook (payload ∩ zips_df)
zip_rows = []
payload_by_zip = {}
for st, payloads in state_payloads.items():
    if st not in state_cutoffs:
        continue
    for p in payloads:
        z = int(p["zip"])
        if z not in zip_lookup.index:
            continue
        qualified = float(zip_lookup.loc[z, "count_qualified"])
        payload_by_zip[z] = p
        zip_rows.append(
            {
                "zip": z,
                "state_code": st,
                "count_qualified": qualified,
                "neat_prop_raw": float(target_prop_global.get(z, 0.0)),
            }
        )

work_meta = pd.DataFrame(zip_rows).drop_duplicates(subset=["zip"], keep="first").reset_index(drop=True)
zip_list = work_meta["zip"].astype(int).tolist()
qualified_by_zip = {int(r.zip): float(r.count_qualified) for r in work_meta.itertuples(index=False)}
state_by_zip = {int(r.zip): r.state_code for r in work_meta.itertuples(index=False)}

s_neat = float(work_meta["neat_prop_raw"].sum())
if s_neat <= 0:
    raise RuntimeError("NEAT proportions sum to 0 over modeled ZIPs; check overlap with federal cache.")

neat_prop = {int(r.zip): float(r.neat_prop_raw) / s_neat for r in work_meta.itertuples(index=False)}

# Status quo: year 5, $0 federal incentive (threshold 0)
sq_panels_raw = {}
yi = STATUS_QUO_YEAR_INDEX
for z in zip_list:
    st = state_by_zip[z]
    cutoff = state_cutoffs[st][yi]
    req = required_incentives(payload_by_zip[z], cutoff)
    adopt = float(np.mean(req <= 0.0))
    sq_panels_raw[z] = max(0.0, adopt * qualified_by_zip[z])

sq_total = float(sum(sq_panels_raw.values()))
if sq_total <= 0:
    raise RuntimeError("Status quo year-5 placements sum to 0; cannot define proportions.")

status_quo_prop = {z: sq_panels_raw[z] / sq_total for z in zip_list}

print(f"Modeled ZIPs: {len(zip_list):,}")
print(f"NEAT prop mass on modeled ZIPs (should be 1.0): {sum(neat_prop.values()):.6f}")
print(f"Status quo year-{STATUS_QUO_YEAR_INDEX + 1} $0-incentive implied panels (unnormalized total): {sq_total:,.0f}")


def compute_objectives_from_placements(placements):
    vals = {}
    for obj in objectives:
        vals[obj.name] = float(obj.calc(zips_df, placements))
    return vals


def placements_for_total(total_panels: float, prop_map: dict):
    t = float(total_panels)
    return {z: t * float(prop_map[z]) for z in zip_list}


panel_totals = np.unique(
    np.concatenate(
        [
            np.array([0]),
            np.linspace(1, MAX_TOTAL_PANELS, N_SWEEP_POINTS),
        ]
    ).astype(np.int64)
)
panel_totals = np.sort(panel_totals)

rows = []
for T in tqdm(panel_totals, desc="Sweep total panels", unit="T"):
    if T <= 0:
        rows.append(
            {
                "total_panels": int(T),
                "scenario": "neat_prop",
                "Carbon Offset": 0.0,
                "Energy Generation": 0.0,
                "Racial Equity": 0.0,
                "Income Equity": 0.0,
            }
        )
        rows.append(
            {
                "total_panels": int(T),
                "scenario": "status_quo_prop",
                "Carbon Offset": 0.0,
                "Energy Generation": 0.0,
                "Racial Equity": 0.0,
                "Income Equity": 0.0,
            }
        )
        continue

    neat_pl = placements_for_total(T, neat_prop)
    sq_pl = placements_for_total(T, status_quo_prop)
    o_neat = compute_objectives_from_placements(neat_pl)
    o_sq = compute_objectives_from_placements(sq_pl)

    rows.append(
        {
            "total_panels": int(T),
            "scenario": "neat_prop",
            "Carbon Offset": o_neat.get("Carbon Offset", np.nan),
            "Energy Generation": o_neat.get("Energy Potential", np.nan),
            "Racial Equity": o_neat.get("Racial Equity", np.nan),
            "Income Equity": o_neat.get("Income Equity", np.nan),
        }
    )
    rows.append(
        {
            "total_panels": int(T),
            "scenario": "status_quo_prop",
            "Carbon Offset": o_sq.get("Carbon Offset", np.nan),
            "Energy Generation": o_sq.get("Energy Potential", np.nan),
            "Racial Equity": o_sq.get("Racial Equity", np.nan),
            "Income Equity": o_sq.get("Income Equity", np.nan),
        }
    )

sweep_df = pd.DataFrame(rows)
display(sweep_df.pivot_table(index="total_panels", columns="scenario", values="Carbon Offset").tail(6))

In [ ]:
metric_cols = ["Carbon Offset", "Energy Generation", "Racial Equity", "Income Equity"]
neat_df = sweep_df[sweep_df["scenario"] == "neat_prop"].sort_values("total_panels")
sq_df = sweep_df[sweep_df["scenario"] == "status_quo_prop"].sort_values("total_panels")

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
axes = axes.flatten()

for ax, m in zip(axes, metric_cols):
    ax.plot(neat_df["total_panels"], neat_df[m], label="NEAT proportions", color="tab:blue", linewidth=2)
    ax.plot(sq_df["total_panels"], sq_df[m], label=f"Status quo (year {STATUS_QUO_YEAR_INDEX + 1}, $0 incentive)", color="tab:orange", linewidth=2, linestyle="--")
    ax.set_xlabel("Total panels (T)")
    ax.set_ylabel(m)
    ax.set_title(f"{m} vs total panels")
    ax.legend(loc="best", fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, MAX_TOTAL_PANELS)

plt.suptitle("Objectives: NEAT ZIP shares vs status-quo ZIP shares (same T)", y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

# Ratio NEAT / status quo (where status quo > 0)
fig2, axes2 = plt.subplots(2, 2, figsize=(12, 9))
axes2 = axes2.flatten()
merged = neat_df.merge(
    sq_df,
    on="total_panels",
    suffixes=("_neat", "_sq"),
)
merged_nonzero = merged[merged["total_panels"] > 0].copy()

for ax, m in zip(axes2, metric_cols):
    num = merged_nonzero[f"{m}_neat"]
    den = merged_nonzero[f"{m}_sq"].replace(0, np.nan)
    ratio = num / den
    ax.plot(merged_nonzero["total_panels"], ratio, color="tab:green", linewidth=2)
    ax.axhline(1.0, color="gray", linestyle=":", alpha=0.8)
    ax.set_xlabel("Total panels (T)")
    ax.set_ylabel(f"NEAT / status quo")
    ax.set_title(f"{m}: ratio")
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, MAX_TOTAL_PANELS)

plt.suptitle("Ratio of objectives (NEAT ÷ status quo) for equal total panels", y=1.02, fontsize=13)
plt.tight_layout()
plt.show()